<a href="https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
# W05 SETUP — Clone the repository

from pathlib import Path

repo_path = Path("/content/flyrank-ml-internship")

if not repo_path.exists():
    !git clone https://github.com/malikasadnadir-max/flyrank-ml-internship.git

print("Repository exists:", repo_path.exists())
print("Repository path:", repo_path)

Repository exists: True
Repository path: /content/flyrank-ml-internship


In [10]:
# W05 SETUP — Check the dataset

from pathlib import Path

data_path = Path(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset exists:", data_path.exists())

if data_path.exists():
    print("Dataset path:", data_path)
    print(
        "Dataset size (MB):",
        round(data_path.stat().st_size / (1024**2), 2)
    )
else:
    print("Dataset was not found.")

Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Dataset size (MB): 6.42


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice and why

I will use **Logistic Regression** as the first learned model.

Logistic Regression fits this lane because the task is binary classification: identifying whether a page is observed as declining or not declining. It is also simple and interpretable, so the model coefficients can help explain which observable features are associated with the decision.

I will compare the model with the Week-4 hand-written baseline using the same target and the same evaluation metric. I will not use `trend_direction`, `trend_pct`, or the label itself as model features because these fields would leak the outcome into the model.

The goal is not to reward model complexity. The goal is to test whether a simple learned model provides better decision-support ranking than the transparent Week-4 rule.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 1 — Load data and prepare the modeling dataset

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

repo_root = Path("/content/flyrank-ml-internship")
data_path = repo_root / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

# Define the binary target.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

# Features available before the outcome.
feature_columns = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "word_count",
]

# Keep only features that actually exist.
missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

if missing_features:
    raise ValueError(f"Missing feature columns: {missing_features}")

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()

# Convert numeric features safely.
for col in feature_columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

print("\nFeatures used:")
for col in feature_columns:
    print(" -", col)

print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\nTarget rate:", round(y.mean(), 4))

# Explicit leakage check.
forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

used_forbidden = forbidden_features.intersection(X.columns)

if used_forbidden:
    raise ValueError(
        f"Leakage detected. Forbidden features found: {used_forbidden}"
    )

print("\nLeakage check: PASS")
print("No label-derived fields are included in X.")

Dataset loaded successfully.
Shape: (30000, 44)

Features used:
 - impressions_90d
 - clicks_90d
 - ctr
 - avg_position
 - days_since_last_update
 - content_age_days
 - word_count

Target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Target rate: 0.5421

Leakage check: PASS
No label-derived fields are included in X.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a **grouped train/test split by client**.

Approximately 80% of clients will be used for training and 20% will be held out for testing. This prevents pages from the same client appearing in both sets.

This is a more honest evaluation for this dataset because the model should be evaluated on clients it did not see during training. The Week-4 baseline will be evaluated on this exact same test set so that the comparison is fair.

The split is fixed with a random seed so the experiment is reproducible.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 2 — Client-grouped train/test split

if "client_id" not in df.columns:
    raise ValueError("client_id column is required for grouped splitting.")

groups = df["client_id"].astype(str)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

overlap = train_clients.intersection(test_clients)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTrain clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nClient overlap:", len(overlap))

if len(overlap) != 0:
    raise ValueError("Client leakage detected between train and test.")

print("Grouped split check: PASS")

print("\nTrain target rate:", round(y_train.mean(), 4))
print("Test target rate:", round(y_test.mean(), 4))

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Client overlap: 0
Grouped split check: PASS

Train target rate: 0.5501
Test target rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train + compare vs my baseline

I will train **Logistic Regression** using the grouped training set and evaluate it on the held-out clients.

The primary ranking metric will be **Precision@50**, matching the ranking-oriented evaluation used in the earlier work. Precision@50 measures the proportion of declining pages among the 50 highest-ranked pages.

The Week-4 baseline and Logistic Regression will be evaluated on the **same test rows**, using the same target and the same metric. This makes the comparison directly interpretable.

The model score is a decision-support ranking score, not a claim about future performance.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 3 — Train Logistic Regression and compare with Week-4 baseline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ---------------------------------------------------------
# 1. Train Logistic Regression
# ---------------------------------------------------------

model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            ),
        ),
    ]
)

model.fit(X_train, y_train)

# Probability of class 1 = declining
model_scores = model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# 2. Precision@K helper
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_idx = np.argsort(-scores, kind="stable")[:k]

    return y_true[top_idx].mean()


# ---------------------------------------------------------
# 3. Logistic Regression ranking metrics
# ---------------------------------------------------------

model_p50 = precision_at_k(
    y_test,
    model_scores,
    k=50
)

model_p100 = precision_at_k(
    y_test,
    model_scores,
    k=100
)


# ---------------------------------------------------------
# 4. Additional classification metrics
# ---------------------------------------------------------

model_predictions = (
    model_scores >= 0.5
).astype(int)

model_precision = precision_score(
    y_test,
    model_predictions,
    zero_division=0
)

model_recall = recall_score(
    y_test,
    model_predictions,
    zero_division=0
)

model_f1 = f1_score(
    y_test,
    model_predictions,
    zero_division=0
)

model_auc = roc_auc_score(
    y_test,
    model_scores
)


# ---------------------------------------------------------
# 5. Recreate the Week-4 baseline on EXACTLY the test rows
# ---------------------------------------------------------

test_df = df.iloc[test_idx].copy()

baseline_stale = (
    (test_df["days_since_last_update"] >= 180)
    & (test_df["impressions_90d"] >= 500)
)

baseline_low_ctr = (
    (test_df["impressions_90d"] >= 500)
    & (test_df["avg_position"] > 0)
    & (test_df["avg_position"] <= 20)
    & (test_df["ctr"] < 0.5)
)

test_df["baseline_score"] = (
    baseline_stale.astype(int) * 2
    + baseline_low_ctr.astype(int)
)


# ---------------------------------------------------------
# 6. Stable baseline ranking
# ---------------------------------------------------------
# Use the stable Week-4 baseline ranking directly.
baseline_ranked = test_df.sort_values(
    "baseline_score",
    ascending=False,
    kind="stable"
)

baseline_y_ranked = (
    baseline_ranked["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
    .to_numpy()
)

baseline_p50 = baseline_y_ranked[:50].mean()
baseline_p100 = baseline_y_ranked[:100].mean()


# ---------------------------------------------------------
# 7. Comparison table
# ---------------------------------------------------------

comparison = pd.DataFrame(
    {
        "method": [
            "Week-4 baseline",
            "Logistic Regression",
        ],
        "Precision@50": [
            baseline_p50,
            model_p50,
        ],
        "Precision@100": [
            baseline_p100,
            model_p100,
        ],
    }
)


print("=" * 80)
print("MODEL VS BASELINE")
print("=" * 80)

print(
    comparison.to_string(
        index=False,
        formatters={
            "Precision@50": "{:.3f}".format,
            "Precision@100": "{:.3f}".format,
        },
    )
)


# ---------------------------------------------------------
# 8. Additional Logistic Regression metrics
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("LOGISTIC REGRESSION ADDITIONAL METRICS")
print("=" * 80)

print("Precision:", round(model_precision, 3))
print("Recall:", round(model_recall, 3))
print("F1:", round(model_f1, 3))
print("ROC-AUC:", round(model_auc, 3))


# ---------------------------------------------------------
# 9. Difference from Week-4 baseline
# ---------------------------------------------------------

p50_difference = model_p50 - baseline_p50

print("\n" + "=" * 80)
print("PRECISION@50 DIFFERENCE")
print("=" * 80)

print(
    "Logistic Regression - Week-4 baseline:",
    round(p50_difference, 3),
)


# ---------------------------------------------------------
# 10. Feature coefficients
# ---------------------------------------------------------

logistic_model = model.named_steps[
    "logistic_regression"
]

coefficients = pd.DataFrame(
    {
        "feature": feature_columns,
        "coefficient": logistic_model.coef_[0],
    }
)

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("\n" + "=" * 80)
print("FEATURE COEFFICIENTS")
print("=" * 80)

print(
    coefficients[
        ["feature", "coefficient"]
    ].to_string(index=False)
)

MODEL VS BASELINE
             method Precision@50 Precision@100
    Week-4 baseline        0.540         0.510
Logistic Regression        0.600         0.530

LOGISTIC REGRESSION ADDITIONAL METRICS
Precision: 0.548
Recall: 0.583
F1: 0.565
ROC-AUC: 0.546

PRECISION@50 DIFFERENCE
Logistic Regression - Week-4 baseline: 0.06

FEATURE COEFFICIENTS
               feature  coefficient
      content_age_days    -0.381335
days_since_last_update     0.226601
                   ctr    -0.182876
            clicks_90d    -0.169395
            word_count     0.063982
       impressions_90d     0.022375
          avg_position    -0.018006


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

I will inspect the highest-ranked pages and the model's classification errors rather than relying only on the overall metric.

False positives are pages the model classifies as declining even though the observed label is not declining. False negatives are declining pages that the model does not classify as declining at the 0.5 threshold.

I will also inspect the Logistic Regression coefficients to understand which observable features the model relies on most. These coefficients describe associations in this dataset; they do not establish causation.

The main limitation is that the model only uses the available page-level signals. Other search, content, or business context may explain errors that are not represented in these features.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 4 — Error analysis and interpretation

test_results = test_df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "content_age_days",
        "word_count",
    ]
].copy()

# Add actual labels and model scores
test_results["actual_label"] = y_test.values
test_results["model_score"] = model_scores

# Classification at the 0.5 threshold
test_results["predicted_label"] = (
    test_results["model_score"] >= 0.5
).astype(int)

# Identify error types
test_results["error_type"] = np.select(
    [
        (test_results["predicted_label"] == 1)
        & (test_results["actual_label"] == 0),

        (test_results["predicted_label"] == 0)
        & (test_results["actual_label"] == 1),
    ],
    [
        "False Positive",
        "False Negative",
    ],
    default="Correct",
)

# ---------------------------------------------------------
# 1. Error counts
# ---------------------------------------------------------

print("=" * 80)
print("ERROR COUNTS")
print("=" * 80)

print(
    test_results["error_type"]
    .value_counts()
)


# ---------------------------------------------------------
# 2. Top 10 model-ranked pages
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("TOP 10 MODEL-RANKED PAGES")
print("=" * 80)

top10_model = (
    test_results
    .sort_values(
        "model_score",
        ascending=False,
        kind="stable"
    )
    .head(10)
)

print(
    top10_model[
        [
            "content_id",
            "model_score",
            "actual_label",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update",
        ]
    ].to_string(index=False)
)


# ---------------------------------------------------------
# 3. Top false positives
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("TOP FALSE POSITIVES")
print("=" * 80)

false_positives = (
    test_results[
        test_results["error_type"] == "False Positive"
    ]
    .sort_values(
        "model_score",
        ascending=False,
        kind="stable"
    )
    .head(10)
)

if len(false_positives) == 0:
    print("No false positives at the 0.5 classification threshold.")
else:
    print(
        false_positives[
            [
                "content_id",
                "model_score",
                "impressions_90d",
                "ctr",
                "avg_position",
                "days_since_last_update",
            ]
        ].to_string(index=False)
    )


# ---------------------------------------------------------
# 4. Top false negatives
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("TOP FALSE NEGATIVES")
print("=" * 80)

false_negatives = (
    test_results[
        test_results["error_type"] == "False Negative"
    ]
    .sort_values(
        "model_score",
        ascending=False,
        kind="stable"
    )
    .head(10)
)

if len(false_negatives) == 0:
    print("No false negatives at the 0.5 classification threshold.")
else:
    print(
        false_negatives[
            [
                "content_id",
                "model_score",
                "impressions_90d",
                "ctr",
                "avg_position",
                "days_since_last_update",
            ]
        ].to_string(index=False)
    )


# ---------------------------------------------------------
# 5. Feature interpretation
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("FEATURE INTERPRETATION")
print("=" * 80)

top_feature = coefficients.iloc[0]["feature"]
top_coefficient = coefficients.iloc[0]["coefficient"]

print(
    f"Largest absolute coefficient: {top_feature}"
)

print(
    f"Coefficient: {top_coefficient:.4f}"
)

print(
    "\nA positive coefficient is associated with higher "
    "predicted probability of the declining class, while "
    "a negative coefficient is associated with lower "
    "predicted probability, holding the other model features "
    "constant."
)


# ---------------------------------------------------------
# 6. Observed model vs baseline result
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVED MODEL VS BASELINE RESULT")
print("=" * 80)

print(
    f"Week-4 baseline Precision@50: {baseline_p50:.3f}"
)

print(
    f"Logistic Regression Precision@50: {model_p50:.3f}"
)

print(
    f"Measured difference: {model_p50 - baseline_p50:+.3f}"
)

print(
    "\nThis result is measured on the held-out client groups "
    "and should be interpreted as directional decision-support "
    "evidence."
)

ERROR COUNTS
error_type
Correct           3335
False Positive    1516
False Negative    1312
Name: count, dtype: int64

TOP 10 MODEL-RANKED PAGES
          content_id  model_score  actual_label  impressions_90d  ctr  avg_position  days_since_last_update
content_829c68e60d13     0.728466             1              307 0.00           4.9                     106
content_9b28cf5ae4f3     0.728281             1             1620 0.06           1.1                     106
content_26d48a980581     0.728216             0             1266 0.00           4.6                     106
content_c65ee459f729     0.727083             1             6526 0.00          17.4                     106
content_b08562686d22     0.726588             1             2846 0.14           2.0                     106
content_df1e8cee858c     0.726299             1             9188 0.09           4.9                     106
content_dcd38075ec2c     0.724938             0                3 0.00           6.0               

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

* [x] Method choice is documented and justified.
* [x] The train/test split is grouped by `client_id`.
* [x] There is no client overlap between training and testing data.
* [x] Logistic Regression is compared with the Week-4 baseline on the same held-out test rows.
* [x] Precision@50 is used as the primary ranking metric.
* [x] Additional metrics are reported for context.
* [x] Model errors and feature coefficients are inspected.
* [x] Label-derived and future-outcome fields are excluded from the model features.
* [x] No client names, private queries, or sensitive URLs are included in the analysis.
* [x] Results are described as observed/measured decision-support evidence rather than as causal claims.
* [x] The experiment uses a reproducible random seed.


In [15]:
# SECTION 5 — Final self-check

print("=" * 80)
print("ML-08 SELF-CHECK")
print("=" * 80)

checks = {
    "Dataset loaded": len(df) == 30000,
    "Seven modeling features available": len(feature_columns) == 7,
    "Target created": "is_declining_label" in df.columns,
    "Grouped split used": True,
    "No client overlap": len(overlap) == 0,
    "Train/test rows exist": len(X_train) > 0 and len(X_test) > 0,
    "Model trained": hasattr(model, "predict_proba"),
    "Precision@50 calculated": not pd.isna(model_p50),
    "Baseline Precision@50 calculated": not pd.isna(baseline_p50),
    "No forbidden features used": len(used_forbidden) == 0,
}

for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status}: {check_name}")

if not all(checks.values()):
    raise ValueError("One or more ML-08 self-checks failed.")

print("\n" + "=" * 80)
print("FINAL OBSERVED RESULTS")
print("=" * 80)

print(f"Test rows: {len(X_test)}")
print(f"Test clients: {len(test_clients)}")
print(f"Week-4 baseline Precision@50: {baseline_p50:.3f}")
print(f"Logistic Regression Precision@50: {model_p50:.3f}")
print(f"Measured Precision@50 difference: {model_p50 - baseline_p50:+.3f}")

print("\nAll ML-08 self-checks: PASS")

ML-08 SELF-CHECK
PASS: Dataset loaded
PASS: Seven modeling features available
PASS: Target created
PASS: Grouped split used
PASS: No client overlap
PASS: Train/test rows exist
PASS: Model trained
PASS: Precision@50 calculated
PASS: Baseline Precision@50 calculated
PASS: No forbidden features used

FINAL OBSERVED RESULTS
Test rows: 6163
Test clients: 7
Week-4 baseline Precision@50: 0.540
Logistic Regression Precision@50: 0.600
Measured Precision@50 difference: +0.060

All ML-08 self-checks: PASS
